# **Problem Statement**

## **Business Context**

Aeolus Renewables is an independent power producer operating a fleet of 1,150 onshore wind turbines (2.5 MW class) across 16 wind farms in the central plains region, with a combined installed capacity of roughly 2.87 GW. The company sells the electricity its turbines produce to the grid under long-term power purchase agreements, where revenue is tied directly to how much energy is delivered - so every hour a turbine is offline is energy that cannot be sold. A regional operations centre monitors the fleet around the clock, supported by an O&M organisation of about 140 field technicians.

Each turbine continuously streams condition data through its SCADA (Supervisory Control and Data Acquisition) system - drivetrain vibration, bearing and oil temperatures, rotor and generator speed, and power output - recorded as 10-minute averages, the standard logging resolution for utility-scale turbines. The exact channels vary by turbine type, but together they describe the health of the main subassemblies in near-real time. Today, maintenance runs on a mix of fixed-interval inspections and reactive repair: technicians follow a scheduled service calendar, and the control room responds when an automated alarm trips or a turbine faults offline.

The most consequential components are in the drivetrain - the gearbox and main bearing - which are expensive, slow to procure, and require a mobile crane to replace. The gearbox alone represents approximately 13% of the overall capital cost of an onshore turbine, and within gearboxes the failures are dominated by bearings: one widely-cited breakdown puts the split at bearings (70%), gears (26%) and other causes (4%). The core problem is recognition. By the time a drivetrain fault has developed far enough to matter, its signature is real but still tangled in normal operating noise across many channels - and the existing fixed thresholds only trip once the fault is near-catastrophic, when the turbine is already offline or the component has seized. A genuinely fault drivetrain can run for hours or days looking "normal" to a threshold alarm, while a control-room analyst has no practical way to tell it apart from a healthy machine reacting to gusty wind. The consequences:

- Each unplanned drivetrain failure takes a turbine offline for an estimated 7–21 days, directly forfeiting saleable energy under the power purchase agreement.
- Emergency crane mobilisation and expedited parts run at a steep premium over the same work scheduled in advance, making an unplanned gearbox replacement a major cost event.
- A degrading main bearing left running frequently destroys the gearbox it feeds, converting a contained repair into a far larger one.
- Control-room analysts manually scan a flood of channels across 1,150 turbines, and alarm fatigue means genuine degradation signals slip through unnoticed until the machine faults offline.


## **Objective**

This proof of concept builds a drivetrain-condition classifier that reads each turbine's live sensor signature and labels the drivetrain as "fault" or "normal", serving operations-centre analysts and maintenance planners. The solution

- Reads each turbine's multi-channel sensor signature and produces a clear fault-vs-normal signal, so analysts can concentrate on the handful of machines genuinely in a fault condition rather than scanning the whole fleet.
- Distinguishes a truly fault drivetrain from normal operating noise more reliably than fixed thresholds, catching faults that are present but not yet severe enough to trip a catastrophic alarm - the window in which a turbine can still be stopped before a bearing fault cascades into gearbox destruction.
Is deliberately tuned to favour catching true failures over avoiding false alerts, because a missed fault drivetrain costs far more than an unnecessary inspection.
- Establishes a measurable detection baseline on historical fleet data, so the capability's accuracy and its operational value can be judged on evidence before any wider rollout.

Once proven at proof-of-concept scale, this capability would give Aeolus a defensible basis to move drivetrain maintenance from reactive repair toward condition-based intervention - reducing unplanned downtime, protecting saleable energy revenue, and containing repairs before they escalate across a 1150-turbine fleet.

## **Data Dictionary**

The dataset, `<name>.csv` contains 10-minute SCADA records for 15 turbines.

### Identifiers & Metadata

| Column | Data Type | Description |
| --- | --- | --- |
| timestamp | datetime | Date and time of the 10-minute logging interval; establishes chronological sequence. |
| turbine_id | object (categorical) | Unique code identifying individual wind turbines; used to track specific asset history. |

### Environmental Conditions

| Column | Data Type | Description |
| --- | --- | --- |
| rated_power_kW | float | Maximum engineered power capacity of the turbine; defines its performance baseline. |
| wind_speed_mps | float | Velocity of the incoming wind; the primary driver of kinetic energy input. |
| wind_direction_deg | float | Compass direction of oncoming wind; used to assess turbine alignment. |
| turbulence_intensity | float | Measure of wind speed fluctuation; higher intensity increases structural fatigue. |
| air_density_kgm3 | float | Mass of air per unit volume; directly impacts aerodynamic lift and power potential. |
| ambient_temp_C | float | Outdoor air temperature surrounding the turbine; affects cooling efficiency. |
| humidity_pct | float | Relative moisture level in the air; flags risks for electrical insulation degradation or corrosion. |

### Operational Control & State

| Column | Data Type | Description |
| --- | --- | --- |
| power_output_kW | float | Real-time electricity generated; drops or fluctuations can signal mechanical drag. |
| rotor_speed_rpm | float | Rotational speed of the main blades; reflects low-speed shaft dynamics. |
| generator_speed_rpm | float | Rotational speed of the generator shaft; crucial for detecting gearbox slip. |
| blade_pitch_angle_deg | float | Angle of the blades relative to the wind; adjusted to control power and rotor speed. |
| yaw_misalignment_deg | float | Angle deviation between wind direction and nacelle orientation; high values cause uneven drivetrain stress. |

### Thermal Metrics (Component Health)

| Column | Data Type | Description |
| --- | --- | --- |
| gearbox_oil_temp_C | float | Temperature of the lubricating oil; spikes indicate excessive mechanical friction. |
| gearbox_bearing_temp_C | float | Internal temperature of gearbox bearings; a leading indicator of bearing wear. |
| generator_bearing_temp_C | float | Temperature of generator bearings; flags alignment or lubrication issues. |
| generator_winding_temp_C | float | Temperature of internal electrical coils; spikes indicate electrical overload or cooling failure. |
| main_bearing_temp_C | float | Temperature of the primary low-speed shaft bearing; handles massive structural loads. |
| nacelle_temp_C | float | Air temperature inside the enclosed housing; reflects global internal heat dissipation. |

### Vibration & Diagnostics (FFT)

| Column | Data Type | Description |
| --- | --- | --- |
| drivetrain_vibration_rms_mmps | float | Overall energy of drivetrain vibrations; general indicator of mechanical roughness. |
| tower_vibration_mmps | float | Structural oscillation of the turbine tower; flags aerodynamic or rotor imbalance. |
| vib_fft_bearing_bpfo | float | Vibration amplitude at the bearing outer-race defect frequency; tracks outer-ring pitting. |
| vib_fft_bearing_bpfi | float | Vibration amplitude at the bearing inner-race defect frequency; tracks inner-ring pitting. |
| vib_fft_gearmesh | float | Vibration amplitude at the teeth-meshing frequency; isolates gearbox tooth wear or misalignment. |
| vib_fft_sideband | float | Vibration amplitude surrounding main frequencies; flags localized faults like cracked gear teeth. |

### Lubrication & Particle Analysis

| Column | Data Type | Description |
| --- | --- | --- |
| oil_particle_count | float | Quantity of metallic debris suspended in lubricating oil; direct indicator of component wear. |
| oil_pressure_bar | float | Pressure of the lubrication system; drops indicate leaks, pump failures, or oil thinning. |

### Asset Lifecycle & History

| Column | Data Type | Description |
| --- | --- | --- |
| operating_hours_total | float | Cumulative runtime of the turbine; represents the asset's total mechanical mileage. |
| cumulative_energy_MWh | float | Total historical electricity produced; measures the lifetime work done by the drivetrain. |
| load_cycles | int | Total count of fatigue-inducing stress variations; correlates directly with structural aging. |
| hours_since_last_maintenance | float | Time elapsed since last service; crucial for identifying maintenance-cycle fatigue. |
| prior_fault_count | int | Total historical fault events triggered; highlights chronic or poorly repaired issues. |
| component_age_days | float | Elapsed lifetime of active components; captures chronological wear independent of runtime. |

### Targets (Labels)

| Column | Data Type | Description |
| --- | --- | --- |
| failure | int (0/1) | The target variable; boolean flag indicating if the drivetrain or turbine is healthy (0) or failing (1). |

# **Please read the instructions carefully before starting the project.**

This is a template Python notebook file in which high-level instructions and tasks to be performed are mentioned, along with pre-filled code blocks in certain sections.

* Feel free to conduct the analysis and build and evaluate the predictive models using AI to generate the necessary code or writing the necessary code from scratch yourself.
* For the code blocks with pre-filled code, please feel free to
    * leverage the pre-filled code blocks as they are, or
    * update the pre-filled code blocks to incorporate necessary changes as per your desired solution workflow for the business problem at hand, or
    * discard the pre-filled code blocks and write the entire code from scratch
* Notebook sections and pre-filled code blocks that contain instructions and tasks to be performed are mentioned.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in markdown cells under the respective notebook sections as per the grading rubric requirements.

# **Installing and Importing the Necessary Libraries**

In [2]:
!pip3 install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 tensorflow==2.20.0 keras==3.13.2 xgboost==3.2.0 seaborn==0.13.2 matplotlib==3.10.0 -q

^C
ERROR: Operation cancelled by user


In [1]:
import seaborn as sns
print("Seaborn version:", sns.__version__)

Seaborn version: 0.13.2


**Note**:
- After running the above cell, kindly restart the notebook kernel (for VS Code) or runtime (for Google Colab), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Standard libraries for tracking execution time and vector/matrix operations
import time
import numpy as np

# Data manipulation and visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tree-based and ensemble machine learning classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Utilities for handling class imbalance, model evaluation, and metric calculations
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# Deep learning framework and specific layers for building Artificial Neural Networks (ANNs)
import tensorflow as tf
from keras.models import Sequential  # Model for building NN sequentially.
from keras.layers import Dense, Dropout, BatchNormalization

# Preprocessing tools for scaling data and tools for model optimization/explainability
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV  # To tune different models
from sklearn.inspection import permutation_importance

# Configuration settings to suppress warnings and format data display outputs
import warnings
warnings.filterwarnings('ignore')              # Suppress warnings for cleaner output logs
sns.set_style('whitegrid')                     # Set a consistent clean grid style for plots
pd.set_option('display.max_columns', None)     # Prevent truncation of columns when displaying dataframes

# **Loading the Data**

In [ ]:
# uncomment and run the below code snippets if you're using Google Colab and the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# **Data Overview**

## Viewing the first and last 5 rows of the dataset

Examine the first and last five rows of the dataset. Identify the available features, observe the values stored in each column, and become familiar with the dataset's structure.

## Checking the shape of the dataset

Identify the total number of rows and columns to understand the size of the dataset before proceeding with the analysis.

## Checking the attribute types

Identify the data type of each feature and determine whether it is numerical, categorical, or datetime. This helps in selecting appropriate preprocessing and analysis techniques.

## Checking the statistical summary

Examine the statistical summary of the numerical features. Observe the key statistics to understand the data distribution and identify any unusual values.

## Checking for missing values

Check for missing values in each feature. Identify which columns contain missing values and the extent of missing data.

## Checking for duplicate values

Check for duplicate records in the dataset. Identify whether any duplicate rows are present before proceeding with the analysis.

# **Data Preprocessing**

- Convert the timestamp column to a datetime format. This enables time-based analysis and feature engineering.
- Extract the day of the week from the timestamp. This helps capture weekly patterns in the data.
- Extract the hour from the timestamp. This helps capture hourly patterns in the data.

# **Exploratory Data Analysis**

## Utility Functions

In [ ]:
def histogram(data_df, col, title='Histogram', xlabel=None, ylabel='Frequency'):
    plt.figure(figsize=(9, 3.6))
    sns.histplot(data_df[col], bins=50, kde=True)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else col)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

def barchart(data_df, x_col, y_col=None, title='Bar Chart', xlabel=None, ylabel=None, rot_degrees=30):
    plt.figure(figsize=(10, 6))
    if y_col: # Bivariate bar chart (x vs y)
        sns.barplot(x=x_col, y=y_col, data=data_df)
    else: # Univariate count plot
        order = data_df[x_col].value_counts().index
        sns.countplot(data=data_df, x=x_col, order=order)

    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else ('Count' if not y_col else y_col))
    plt.xticks(rotation=rot_degrees, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def boxplot(data_df, x_col, y_col, title='Box Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.boxplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

def scatterplot(data_df, x_col, y_col, title='Scatter Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.scatterplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

## Univariate Analysis

### `failure`

In [ ]:
barchart(data, 'failure', title='Target Distribution: Drivetrain Condition', xlabel='failure (0 = normal, 1 = fault)')

### ```wind_speed_mps```

Generate a histogram of the `wind_speed_mps` feature. Identify the range where most observations are concentrated and check for any skewness or unusual values.

### ```air_density_kgm3```

Generate a histogram of the `air_density_kgm3` feature. Observe where most air density values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```gearbox_oil_temp_C```

Generate a histogram of the `gearbox_oil_temp_C` feature. Observe where most gearbox oil temperature values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```generator_winding_temp_C```

Generate a histogram of the `generator_winding_temp_C` feature. Observe where most generator winding temperature values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```drivetrain_vibration_rms_mmps```

Generate a histogram of the `drivetrain_vibration_rms_mmps` feature. Observe where most vibration values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```tower_vibration_mmps```

Generate a histogram of the `tower_vibration_mmps` feature. Observe where most tower vibration values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```oil_particle_count```

Generate a histogram of the `oil_particle_count` feature. Observe where most oil particle count values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```prior_fault_count```

Generate a histogram of the `prior_fault_count` feature. Observe where most prior fault count values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### ```component_age_days```

Generate a histogram of the `component_age_days` feature. Observe where most component age values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

## Bivariate Analysis

### ```turbine_id``` vs ```failure```

In [ ]:
barchart(data_df=data, x_col='turbine_id', y_col='failure', title='Failure Rate Across Turbines', xlabel='Turbine ID', ylabel='Failure Rate (0 = normal, 1 = fault)')

### `wind_speed_mps` vs `power_output_kW`

Perform the following steps to analyze the relationship between wind speed and power output:

1. Group the `wind_speed_mps` values into intervals (bins).
2. Calculate the average `power_output_kW` for each wind speed interval.
3. Plot the average power output against the wind speed intervals.
4. Observe how power output changes as wind speed increases and identify any trends or patterns.

### ```rotor_speed_rpm``` vs ```generator_speed_rpm```

Generate a scatter plot using `rotor_speed_rpm` and `generator_speed_rpm`. Observe whether a relationship exists between the two variables, identify the overall trend, and look for any unusual observations or outliers.

### ```wind_speed_mps``` vs ```rotor_speed_rpm```

Generate a scatter plot using `wind_speed_mps` and `rotor_speed_rpm`. Observe how rotor speed changes with increasing wind speed, identify the overall relationship between the two variables, and look for any unusual observations or outliers.

### `drivetrain_vibration_rms_mmps` vs `failure`

Generate a box plot of `drivetrain_vibration_rms_mmps` across the `failure` classes. Compare the distribution of vibration levels between normal and fault conditions, and identify any differences in central tendency, spread, and potential outliers.

### `oil_particle_count` vs `failure`

Generate a box plot of `oil_particle_count` across the `failure` classes. Compare the distribution of oil particle counts between normal and fault conditions, and identify any differences in central tendency, spread, and potential outliers.

### `gearbox_bearing_temp_C` vs `failure`

Generate a box plot of `gearbox_bearing_temp_C` across the `failure` classes. Compare the distribution of gearbox bearing temperatures between normal and fault conditions, and identify any differences in central tendency, spread, and potential outliers.

### `power_output_kW` vs `failure`

Generate a box plot of `power_output_kW` across the `failure` classes. Compare the distribution of power output between normal and fault conditions, and identify any differences in central tendency, spread, and potential outliers.

### `prior_fault_count` vs `failure`

Generate a box plot of `prior_fault_count` across the `failure` classes. Compare the distribution of prior fault counts between normal and fault conditions, and identify any differences in central tendency, spread, and potential outliers.

### ```failure``` vs ```hour```

Perform the following steps to analyze how failure rates vary throughout the day:

1. Calculate the failure rate for each hour of the day.
2. Convert the failure rate into percentage values for easier interpretation.
3. Generate a bar chart to visualize the hourly failure rates.
4. Identify the hours with the highest and lowest failure rates, and observe any time-based patterns in failures.

### ```failure``` vs ```dayofweek```

Perform the following steps to analyze how failure rates vary across the days of the week:

1. Calculate the failure rate for each day of the week.
2. Convert the failure rate into percentage values for easier interpretation.
3. Arrange the days in chronological order from Monday to Sunday.
4. Generate a bar chart to visualize the failure rates across the week.
5. Identify the days with the highest and lowest failure rates, and observe any weekly patterns in failures.

## Multivariate Analysis

### Correlation Analysis

Create a list of all numerical features that will be used for correlation analysis. These features will be used to examine relationships between numerical variables and identify patterns that may be useful for further analysis and model building.

Generate a correlation matrix for the numerical features. Examine the strength and direction of relationships between variables, identify highly correlated feature pairs, and look for potential multicollinearity that may affect model performance.

# **Data Preprocessing**

## Splitting the data into train, validation, and test sets

This data is **panel time-series**: 15 turbines, each logged every 10 minutes over two months, and a single degradation episode spans many consecutive rows. A random split would scatter rows from the same episode across train and test, letting the model effectively see failures it is later scored on - inflating every metric.


In [ ]:
data = data.sort_values(by=['timestamp','turbine_id'])

In [ ]:
n = len(data)

In [ ]:
# Uncomment the below train and validation indices to create a 70% train, 15% validation, and 15% test split.
# i_train = int(n * 0.70)
# i_val = int(n * 0.85)

# Uncomment the below train and validation indices to create a 90% train, 5% validation, and 5% test split.
# i_train = int(n * 0.90)
# i_val = int(n * 0.95)

# Uncomment the below train and validation indices to create a 80% train, 10% validation, and 10% test split.
# i_train = int(n * 0.80)
# i_val = int(n * 0.90)

In [ ]:
data_train = data.iloc[:i_train]
data_val = data.iloc[i_train:i_val]
data_test = data.iloc[i_val:]

X_train = data_train.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_train = data_train['failure']

X_valid = data_val.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_valid = data_val['failure']

X_test = data_test.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_test = data_test['failure']

In [ ]:
print(f'Train split failure rate: {y_train.mean()*100:.2f}%')
print(f'Validation split failure rate: {y_valid.mean()*100:.2f}%')
print(f'Test split failure rate: {y_test.mean()*100:.2f}%')

## Missing Value Treatment

### Display the percentage of missing values

Check the percentage of missing values in the training, validation, and test datasets. Identify the features with missing values before selecting an appropriate imputation strategy.

### `gearbox_oil_temp_C`

Perform the following steps to handle missing values in the `gearbox_oil_temp_C` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.
2. Calculate the selected statistic using only the training data.
3. Use the calculated value to fill the missing values in the training, validation, and test datasets.
4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### ```generator_bearing_temp_C```

Perform the following steps to handle missing values in the `generator_bearing_temp_C` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.
2. Calculate the selected statistic using only the training data.
3. Use the calculated value to fill the missing values in the training, validation, and test datasets.
4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### ```oil_pressure_bar```

Perform the following steps to handle missing values in the `oil_pressure_bar` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.
2. Calculate the selected statistic using only the training data.
3. Use the calculated value to fill the missing values in the training, validation, and test datasets.
4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

# **Model Building**

**Note**: All five models provided in the subsequent subsections need to be built and evaluated.

## Model Evaluation Criterion

In [ ]:
# Uncomment one of the following evaluation metrics

# metric_of_choice = 'accuracy'
# metric_of_choice = 'precision'
# metric_of_choice = 'recall'
# metric_of_choice = 'f1'

## Utility Functions

Before moving ahead, we define a function to check the performance of the model using different metrics.

- We will be using metric functions defined in sklearn for accuracy, precision, recall, f1_score
- We will create a function which will print out all the above metrics in one go.

In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier (sklearn or tensorflow)
    predictors: independent variables
    target: dependent variable
    """

    # 1. Get raw predictions
    pred_raw = model.predict(predictors)

    # 2. Check the shape or values to handle DL vs ML models safely
    # If the predictions are probabilities (floats between 0 and 1, not exactly 0 or 1)
    # We check if any value falls strictly between 0 and 1
    if np.issubdtype(pred_raw.dtype, np.floating) and not np.all(np.isin(pred_raw, [0.0, 1.0])):
        # FOR TENSORFLOW DL MODELS: threshold the probabilities at 0.5
        pred = (pred_raw > 0.5).astype(int)
    else:
        # FOR SKLEARN ML MODELS: use them directly
        pred = pred_raw

    # Flatten predictions to ensure they match target shape perfectly
    pred = np.array(pred).flatten()

    # 3. Compute metrics
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    # 4. Creating a dataframe of metrics
    data_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return data_perf

In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    # 1. Get raw predictions
    y_pred_raw = model.predict(predictors)

    # 2. Check if predictions are probabilities (typical for Keras/TF models or some ML models with predict_proba)
    # If y_pred_raw contains float values between 0 and 1, it's likely probabilities.
    if np.issubdtype(y_pred_raw.dtype, np.floating) and np.any((y_pred_raw > 0) & (y_pred_raw < 1)):
        # Binarize probabilities using a threshold (e.g., 0.5)
        y_pred = (y_pred_raw > 0.5).astype(int)
    else:
        # Use predictions directly (typical for scikit-learn classifiers that return binary labels)
        y_pred = y_pred_raw

    # Ensure y_pred is flattened to a 1D array, as confusion_matrix expects this format.
    y_pred = np.array(y_pred).flatten()

    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show() # Added plt.show() to explicitly display the plot

## Decision Tree Classifier

Perform the following steps to train and evaluate the Decision Tree model:

1. Create a Decision Tree classifier using `class_weight='balanced'` to assign higher importance to the minority class and reduce the impact of class imbalance.
2. Train the model using the training dataset.
3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Random Forest Classifier

Perform the following steps to train and evaluate the Random Forest model:

1. Create a Random Forest classifier using `class_weight='balanced'` to assign higher importance to the minority class and reduce the impact of class imbalance.
2. Train the model using the training dataset.
3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Gradient Boosting Classifier

Perform the following steps to train and evaluate the Gradient Boosting model:

1. Create a Gradient Boosting classifier using the selected random state.
2. Train the model using the training dataset.
3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## XGBoost

Perform the following steps to train and evaluate the XGBoost model:

1. Calculate the number of negative and positive samples in the training data.
2. Compute the `scale_pos_weight` value as the ratio of negative samples to positive samples to address class imbalance.
3. Create an XGBoost classifier using the calculated `scale_pos_weight` and the selected random state.
4. Train the model using the training dataset.
5. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
6. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
7. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Neural Networks(ANN)

In [ ]:
tf.keras.backend.clear_session()

Perform the following steps to standardize the feature values:

1. Create a `StandardScaler` object to standardize the numerical features.
2. Fit the scaler using only the training dataset and transform the training features.
3. Use the same fitted scaler to transform the validation and test datasets.
4. Apply the same scaling parameters across all datasets to ensure a consistent preprocessing strategy and prevent data leakage.

Perform the following steps to build the neural network architecture:

1. Choose the number of neurons for the first hidden layer between **8 and 128**.
   - Fewer neurons (**8–32**) create a simpler model that trains faster and is less likely to overfit.
   - More neurons (**64–128**) enable the model to learn more complex patterns but increase training time and the risk of overfitting.
2. Choose an activation function for the first hidden layer (for example, `relu`, `tanh`, or `sigmoid`) based on the learning behavior you want.
3. Create the output layer with a single neuron and a `sigmoid` activation function to predict the probability of the positive class.

Display the neural network architecture. Review the number of layers, the output shape of each layer, and the total number of trainable parameters before training the model.

Perform the following steps to configure the neural network for training:

1. Choose an optimizer (`sgd` or `adam`) based on the desired training behavior.
   - `sgd` updates the model using simple gradient descent and may require more epochs to converge.
   - `adam` automatically adapts the learning rate, often converges faster, and is a common default choice.
2. Use `binary_crossentropy` as the loss function since this is a binary classification problem.
3. Configure the model to track **Recall**, **Precision**, and **Binary Accuracy** during training to evaluate its classification performance.

Perform the following steps to configure the training process:

1. Choose the number of training epochs between **10 and 100** based on the desired training duration and model performance.
2. Select a batch size between **16 and 128** based on the available computational resources and training behavior.
3. Use the selected values to control how long the model trains and how many samples are processed in each weight update.

In [ ]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weights[i] for i in range(len(weights))}

In [ ]:
start = time.time()
history = model.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) , class_weight=class_weight_dict ,batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

Perform the following steps to evaluate the neural network model:

1. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
2. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
3. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## **Baseline Model Performance Comparison**

### Training performance comparison

Perform the following steps to compare the training performance of all baseline models:

1. Combine the training performance metrics of all baseline models into a single comparison table.
2. Display the consolidated table to compare the performance of each model across the selected evaluation metrics.
3. Use the comparison to identify the best-performing models for hyperparameter tuning.

### Validation performance comparison

Perform the following steps to compare the validation performance of all baseline models:

1. Combine the validation performance metrics of all baseline models into a single comparison table.
2. Display the consolidated table to compare the performance of each model across the selected evaluation metrics.
3. Use the comparison to identify the best-performing models for hyperparameter tuning based on their validation performance.

# **Hyperparameter Tuning**

**Note:** Choose at least two best-performing baseline models across the train and validation sets to proceed with tuning.

## XGBoost

Perform the following steps to define the hyperparameter search space for the XGBoost model:

1. Choose values for **`n_estimators`**, which controls the number of trees in the model. More trees can improve learning but increase training time.
2. Choose values for **`learning_rate`**, which controls how quickly the model learns. Smaller values require more trees, while larger values learn faster but may overshoot.
3. Choose values for **`max_depth`**, which controls the maximum depth of each tree. Deeper trees capture more complex patterns but are more likely to overfit.
4. Choose values for **`min_child_weight`**, which controls the minimum weight required to create a new split. Larger values produce more conservative trees.
5. Choose values for **`gamma`**, which specifies the minimum loss reduction required before making a split. Larger values reduce unnecessary splits.
6. Choose values for **`subsample`**, which determines the fraction of training samples used to build each tree. Lower values improve generalization, while higher values use more data.
7. Choose values for **`colsample_bytree`**, which specifies the fraction of features used to build each tree. Lower values increase diversity among trees.
8. Choose values for **`reg_alpha`** and **`reg_lambda`**, which apply L1 and L2 regularization to reduce overfitting.
9. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the XGBoost model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.
2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.
3. Display the best hyperparameter combination and its corresponding cross-validation score.

1. Retrieve the best-performing XGBoost model identified during the randomized search.
2. Retrain the selected model using the complete training dataset.
3. Use the retrained model for subsequent evaluation on the training, validation, and test datasets.

## Neural Networks

In [ ]:
tf.keras.backend.clear_session()

Perform the following steps to build the tuned neural network architecture:

1. Choose the number of neurons and an activation function for the first hidden layer. Then apply **Batch Normalization** to stabilize training, followed by **Dropout** to reduce overfitting.
2. Choose the number of neurons and an activation function for the second hidden layer. Again, apply **Batch Normalization** followed by **Dropout** to improve generalization.
3. Create the output layer with a single neuron and a **`sigmoid`** activation function to predict the probability of the positive class.
4. Review the selected architecture before proceeding with model training.

Display the tuned neural network architecture. Review the number of layers, the output shape of each layer, and the total number of trainable parameters before training the model.

Perform the following steps to configure the tuned neural network for training:

1. Choose an optimizer (`adam` or `sgd`) based on the desired training behavior.
   - `adam` automatically adapts the learning rate, often converges faster, and is a common default choice.
   - `sgd` updates the model using simple gradient descent and may require more epochs to converge.
2. Use `binary_crossentropy` as the loss function since this is a binary classification problem.
3. Configure the model to track **Recall**, **Precision**, and **Binary Accuracy** during training to evaluate its classification performance.

Perform the following steps to configure the training process:

1. Choose the number of training epochs between **20 and 150** based on the desired training duration and model performance.
2. Select a batch size between **16 and 128** based on the available computational resources and training behavior.
3. Use the selected values to control how long the model trains and how many samples are processed in each weight update.

In [ ]:
start = time.time()
history = model1.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) ,class_weight=class_weight_dict, batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

Perform the following steps to evaluate the tuned neural network model:

1. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.
2. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.
3. Compare the training and validation results to assess whether the tuned architecture improves generalization and reduces overfitting compared to the baseline neural network.

## Decision Tree

Perform the following steps to define the hyperparameter search space for the Decision Tree model:

1. Choose the splitting **`criterion`** (`gini` or `entropy`) to determine how the model selects the best feature at each split.
2. Choose values for **`max_depth`** to control the maximum depth of the tree. Deeper trees can capture more complex patterns but are more likely to overfit.
3. Choose values for **`min_samples_split`** to control the minimum number of samples required to split a node. Larger values create more conservative trees.
4. Choose values for **`min_samples_leaf`** to control the minimum number of samples required in each leaf node. Larger values produce smoother and more generalized trees.
5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split.
6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Decision Tree model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.
2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.
3. Display the best hyperparameter combination and its corresponding cross-validation score.

Display the best Decision Tree model identified during hyperparameter tuning. Review the optimized model configuration before using it for retraining and evaluation.

Evaluate the tuned Decision Tree model on the training dataset and display its performance metrics. Use the results to assess how well the tuned model has learned the training data.

Evaluate the tuned Decision Tree model on the validation dataset and display its performance metrics. Compare these results with the training performance to assess how well the tuned model generalizes to unseen data.

## Random Forest

Perform the following steps to define the hyperparameter search space for the Random Forest model:

1. Choose values for **`n_estimators`**, which controls the number of trees in the forest. More trees generally improve stability but increase training time.
2. Choose values for **`max_depth`** to control the maximum depth of each tree. Deeper trees capture more complex patterns but are more likely to overfit.
3. Choose values for **`min_samples_split`** to control the minimum number of samples required to split a node. Larger values create more conservative trees.
4. Choose values for **`min_samples_leaf`** to control the minimum number of samples required in each leaf node. Larger values improve generalization by preventing overly specific splits.
5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split in each tree.
6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Random Forest model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.
2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.
3. Display the best hyperparameter combination and its corresponding cross-validation score.

Perform the following steps after hyperparameter tuning:

1. Retrieve the best-performing Random Forest model identified during the randomized search.
2. Retrain the selected model using the complete training dataset.
3. Evaluate the retrained model on the training dataset and display its performance metrics.
4. Evaluate the retrained model on the validation dataset and display its performance metrics.
5. Compare the training and validation results to assess how well the tuned model generalizes to unseen data.

## Gradient Boosting

Perform the following steps to define the hyperparameter search space for the Gradient Boosting model:

1. Choose values for **`n_estimators`**, which controls the number of boosting stages. More boosting stages can improve learning but increase training time.
2. Choose values for **`learning_rate`**, which controls how quickly the model learns. Smaller values require more boosting stages, while larger values learn faster but may overfit.
3. Choose values for **`max_depth`** to control the maximum depth of each decision tree. Deeper trees capture more complex patterns but are more likely to overfit.
4. Choose values for **`subsample`**, which determines the fraction of training samples used for each boosting stage. Lower values improve generalization by introducing randomness, while higher values use more data for learning.
5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split in each tree.
6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Gradient Boosting model using randomized search:

1. Perform randomized search with **3-fold cross-validation** using the selected evaluation metric.
2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.
3. Display the best hyperparameter combination and its corresponding cross-validation score.

Perform the following steps after hyperparameter tuning:

1. Retrieve the best-performing Gradient Boosting model identified during the randomized search.
2. Retrain the selected model using the complete training dataset.
3. Evaluate the retrained model on the training dataset and display its performance metrics.
4. Evaluate the retrained model on the validation dataset and display its performance metrics.
5. Compare the training and validation results to assess how well the tuned model generalizes to unseen data.

# **Final Model Selection**

Perform the following steps to compare the training performance of the tuned models:

1. Add the training performance metrics of the tuned models to the existing comparison table.
2. Display the updated comparison table.
3. Compare the baseline and tuned models to determine whether hyperparameter tuning improved the training performance.

Perform the following steps to compare the validation performance of the tuned models:

1. Add the validation performance metrics of the tuned models to the existing comparison table.
2. Display the updated comparison table.
3. Compare the baseline and tuned models to determine whether hyperparameter tuning improved the validation performance and generalization.

Perform the following steps to select the final model:

1. Review the performance of all baseline and tuned models using the selected evaluation metric.
2. Select the model that best meets the business objective and demonstrates strong generalization on the validation dataset.
3. Assign the selected model as the final model for evaluation on the test dataset.

## Feature Importance

In [ ]:
def plot_feature_importances(model, X, y, feature_names=None, color="violet", figsize=(10, 10)):
    """
    Plots feature importances for both Traditional ML models (XGBoost, RF)
    and Deep Learning models (ANNs).

    Parameters:
    - model: Trained model object (XGBoost, Sklearn, Keras wrapper, etc.)
    - X: Evaluation features (DataFrame or 2D array)
    - y: Evaluation targets (Series or 1D array)
    - feature_names: List of feature names. If None and X is a DataFrame, uses X.columns.
    """
    # 1. Automatically grab feature names if not provided
    if feature_names is None:
        if hasattr(X, 'columns'):
            feature_names = X.columns
        else:
            feature_names = [f"Feature {i}" for i in range(X.shape[1])]

    # Convert X to numpy if it's a DataFrame for permutation calculation safety
    X_val = X.values if hasattr(X, 'columns') else X

    # 2. Extract or calculate importances
    if hasattr(model, 'feature_importances_'):
        print("💡 Detected Tree-based model. Extracting built-in feature importances...")
        importances = model.feature_importances_
        title = "Feature Importances (Tree-based Model)"
        xlabel = "Relative Importance"
    else:
        print("💡 Detected ANN/Black-box model. Calculating Permutation Importance...")
        # n_repeats is how many times a feature is shuffled; higher is more accurate but slower
        result = permutation_importance(model, X_val, y, n_repeats=5, random_state=42)
        importances = result.importances_mean
        title = "Feature Importances (Permutation Importance - ANN)"
        xlabel = "Mean Performance Drop when Shuffled"

    # 3. Sort the importances in ascending order
    indices = np.argsort(importances)

    # 4. Plotting
    plt.figure(figsize=figsize)
    plt.title(title)
    plt.barh(range(len(indices)), importances[indices], color=color, align="center")
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_importances(best_model, X_train, y_train)

## Final Model Test Performance

1. Evaluate the selected model using the unseen test dataset.
2. Display the performance metrics and confusion matrix to measure how well the model generalizes to new data.
3. Use the test results as the final estimate of the model's real-world performance.

# **Business Insights and Recommendations**

## Business Insights

-
-
-

## Recommendations

-
-
-